# The full benchmark on RAW RV — 10 models + HAR-RV × 10 datasets × h = 1, 5, 22

The same grid as `colab_benchmark.ipynb`, on the **variance scale** instead of
`ln(RV)`: every model forecasts the h-day forward mean of RV directly, HAR-RV
is fitted on the same rows without `--log`, and everything is scored there.

**The hyper-parameters are the existing Optuna winners, reused unchanged.**
Nothing is re-tuned. Step 5 takes each `tuning/ProjectC_tuning/<Model>_best.json`
and drops the single token `--log` from its command line; every other flag —
architecture, learning rate, batch size, schedule, `--aggregate_mean`,
`seq_len 96` — is passed through verbatim. That is the whole of the change, and
it is enough, because the modelling scale is a property of the anchor:
`run_benchmark.py` reads `--log`'s presence off the tuned command line, calls
the sweep `ln_RV` or `raw_RV` accordingly, and fits HAR-RV on whichever scale it
finds.

So the two sweeps differ in exactly one variable. Same architectures, same
optimiser settings, same splits, same rows, same target `Y^(h)` — one modelled
as `ln(mean RV)` and the other as `mean RV`. Any difference in the tables is
attributable to the scale, which is what a clean comparison needs.

**The caveat that belongs in the caption.** Those hyper-parameters were selected
by minimising a validation loss on `ln(RV)`. Raw RV is right-skewed with a heavy
tail where `ln(RV)` is close to Gaussian, so a learning rate and schedule that
were a good choice on the first are not necessarily a good choice on the second.
A model that reads badly here may be mistuned for the raw scale rather than
unsuited to it — a statement about the transfer, not about the architecture.
Step 11 and `failures.csv` are where that shows up.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

**This does not finish in one session.** 300 cells × 10 seeds = **3 000
trainings**; a Colab session is capped at ~12 h. Everything is written to
Google Drive and the sweep **resumes** — reconnect, re-run steps 2–5, then run
step 8 again, and the cells already on Drive are skipped. Step 9 shows how far
along you are; step 10's tables work on a half-finished sweep.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > T4 GPU. 3000 trainings on CPU is not realistic.'

## 2. Mount Drive

The raw anchors, the forecasts, the tables and the loss matrices go here so they
survive a disconnect — that is what makes the sweep resumable across sessions.

Both directories are **separate from the log run's**, deliberately. Anchors
carry the scale, so a directory holding both kinds would make
`run_benchmark.py` refuse the sweep ("one scale per sweep"), and a results
directory holding both would produce blocks the aggregation reports as mixed
and skips. Keeping them apart is also what lets the two sweeps be set side by
side afterwards (§12).

Checkpoints do **not** go to Drive: they are rewritten every improving epoch,
and on a Drive mount that would dominate the runtime. They are deleted after
each cell anyway.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ANCHOR_DIR  = '/content/drive/MyDrive/ProjectC_anchors_raw'     # step 5 writes here
RESULTS_DIR = '/content/drive/MyDrive/ProjectC_benchmark_raw'   # the sweep writes here
CKPT_DIR    = '/content/_ckpt'                                  # local disk, deleted per cell

import os
os.makedirs(ANCHOR_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('anchors ->', ANCHOR_DIR)
print('results ->', RESULTS_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is public, so this needs no credentials. Re-running the cell
in a later session updates an existing clone instead of failing.

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/benchmark-raw-rv-7ef3kb'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn, statsmodels and
matplotlib. These are the extras this repo needs — `fast_pytorch_kmeans` for
AdaWaveNet, `reformer-pytorch`/`local-attention` because
`layers/SelfAttention_Family.py` imports them at module load.

No `optuna` here: nothing is searched in this notebook.

In [ ]:
!pip install -q einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 5. Build the raw anchors from the log winners

The one cell that makes this notebook different from `colab_benchmark.ipynb`.
For each of the ten `<Model>_best.json` files it copies the record, deletes the
`--log` token from `command`, and writes it to `ANCHOR_DIR`. It takes a second,
it trains nothing, and it re-runs harmlessly — so run it again after every
reconnect, before the sweep.

Why editing the command line is the whole job: `benchmark_config.load_anchor`
parses that string, drops the flags the orchestrator owns (`--model_id`,
`--des`, `--itr`, `--seed`, `--checkpoints`, `--num_workers`, `--data`,
`--root_path`, `--data_path`, `--pred_len`, `--train_epochs`, `--patience`),
and passes **everything else through unchanged**. `--log` is not one of the
owned flags, which is exactly why it has to be removed here and why removing it
is sufficient. Every hyper-parameter Optuna chose survives untouched.

Two fields are rewritten rather than copied:

* `best_val_loss` is cleared. The number in the log file is a validation loss on
  `ln(RV)` and does not describe this anchor; leaving it would put a wrong value
  in every cell's `anchor_val_loss`. It is kept under `best_val_loss_ln_study`
  so the provenance stays readable.
* `scale_source` records which file this came from, so a `.npz` written months
  from now still says where its configuration was selected.

`SRC` points at the anchors shipped with the repo. Point it at a Drive copy
instead if you have re-run the log study since.

In [ ]:
import glob, json, os, shutil

SRC = 'tuning/ProjectC_tuning'      # the log-scale Optuna winners

paths = sorted(glob.glob(os.path.join(SRC, '*_best.json')))
if not paths:
    raise SystemExit(f'no <Model>_best.json under {SRC}')

for path in paths:
    b = json.load(open(path))
    tokens = b['command'].split()
    if '--log' not in tokens:
        print(f'[skip] {b["model"]}: its command line has no --log to remove')
    b['command'] = ' '.join(t for t in tokens if t != '--log')
    b['best_val_loss_ln_study'] = b.get('best_val_loss')
    b['best_val_loss'] = None       # the ln(RV) number is not this anchor's
    b['scale_source'] = f'{path} with --log removed; hyper-parameters unchanged'
    with open(os.path.join(ANCHOR_DIR, os.path.basename(path)), 'w') as fh:
        json.dump(b, fh, indent=2)

print(f'wrote {len(paths)} raw anchor(s) to {ANCHOR_DIR}')

### Check: same hyper-parameters, one flag fewer

This is the assertion the whole notebook rests on. It re-parses both command
lines the way `benchmark_config` does and prints, per model, the flags that
differ between the log anchor and the raw one. The only admissible answer is
`-- log` and nothing else — the token removed, no value changed, none added.

In [ ]:
import glob, json, os
import pandas as pd

from orchestrate.benchmark_config import ORCHESTRATOR_FLAGS, load_anchor, parse_command

rows = []
for path in sorted(glob.glob(os.path.join(SRC, '*_best.json'))):
    model = os.path.basename(path)[:-len('_best.json')]
    ln = parse_command(json.load(open(path))['command'])
    raw, meta = load_anchor(model, ANCHOR_DIR)          # already flag-stripped
    ln = {k: v for k, v in ln.items() if k not in ORCHESTRATOR_FLAGS}

    added = [f'+ {k}' for k in raw if k not in ln]
    removed = [f'- {k}' for k in ln if k not in raw]
    changed = [f'~ {k}: {" ".join(ln[k])} -> {" ".join(raw[k])}'
               for k in raw if k in ln and ln[k] != raw[k]]
    rows.append({'model': model, 'scale': meta['scale'],
                 'differences': ', '.join(removed + added + changed) or '(none)'})

df = pd.DataFrame(rows)
display(df)

ok = (df['scale'] == 'raw_RV').all() and (df['differences'] == '- log').all()
print('CHECK: ' + ('every anchor is raw_RV and differs from its log twin by '
                   'exactly the --log token'
                   if ok else '!! something other than --log changed — read the table'))

In [ ]:
# the exact command line each benchmark cell will run, minus the per-cell flags
for path in sorted(glob.glob(os.path.join(ANCHOR_DIR, '*_best.json'))):
    b = json.load(open(path))
    print(f'# {b["model"]}\n{b["command"]}\n')

## 6. Validate every configuration (~1 minute, CPU)

Builds all 10 models at all 3 horizons from the raw anchors and pushes one batch
through each, then prints how many forecasts each dataset holds per horizon. A
configuration the architecture rejects should surface here, not at hour six of
the sweep.

`--scale raw_RV` is the guard on the whole notebook: the flag has to agree with
what the anchors say, so a stale `ANCHOR_DIR` — or a step 5 that never ran —
fails here in a second with a message naming both scales.

In [ ]:
!python orchestrate/run_benchmark.py --validate \
    --anchor_dir $ANCHOR_DIR --scale raw_RV

## 7. Smoke test (~1 minute)

Two cheap models plus the baseline on one dataset at one horizon, one seed, 5
epochs — written to local disk, so a smoke cell can never be mistaken for a real
one by the resume logic. If it ends with a summary table headed
`h = 1  [raw_RV]`, the environment is wired up correctly; the scale in that
header is read back off the stored cells, not off the flag.

Two things to expect in the log, both consequences of dropping `--log`:

* the HAR-RV line mentions `har_rv_h01_fitted.csv`, not
  `har_rv_log_h01_fitted.csv` — the baseline is fitted raw here too;
* each cell still prints `Non-positive targets will be dropped`.
  `--aggregate_mean` implies `--drop_nonpositive` in `run.py`, so the raw run
  drops the same zero-RV non-trading days the log run drops and HAR-RV drops
  unconditionally. The two families stay fitted on identical rows — five of the
  forex series carry two such days each.

In [ ]:
!python -u orchestrate/run_benchmark.py \
    --datasets EURUSD --horizons 1 --models DLinear FITS HAR-RV --itr 1 --quick \
    --anchor_dir $ANCHOR_DIR --scale raw_RV \
    --results_dir /content/_smoke_raw --checkpoint_dir $CKPT_DIR 2>&1 | tail -25

## 8. The sweep

3 000 trainings (300 cells × 10 seeds) plus 10 HAR-RV fits. One subprocess per
cell, so an OOM or a CUDA fault costs one cell rather than the run; each cell
writes its forecasts the moment it finishes, and cells already on Drive are
skipped — which is what makes this cell safe to re-run after every disconnect.

Ordered dataset → horizon → model, so an interrupted run leaves **complete**
(dataset, horizon) blocks behind — the unit the MCS is defined over.

`EXTRA` runs a subset, which is the practical way to do this over several
sessions:

* `['--assets', 'crypto']` or `['--datasets', 'EURUSD', 'AUDUSD']`
* `['--horizons', '1']`
* `['--itr', '3']` — 3 repeats per cell instead of 10, roughly a third of the cost
* `['--models', 'FITS', 'DLinear', 'TSLANet']` — the cheap ones first

Watch the failure rate on the first few cells. A learning rate selected on
`ln(RV)` can diverge on a heavy-tailed raw target, and a cell that dies is
recorded in `failures.csv` and skipped rather than retried — so a model failing
across the board is the transfer telling you something, not a bug. `--quick`
on that model at one dataset is the cheap way to look.

In [ ]:
import subprocess, time

EXTRA = []          # e.g. ['--assets', 'crypto'] or ['--itr', '3']

started = time.time()
subprocess.run(['python', '-u', 'orchestrate/run_benchmark.py',
                '--anchor_dir', ANCHOR_DIR,
                '--scale', 'raw_RV',
                '--results_dir', RESULTS_DIR,
                '--checkpoint_dir', CKPT_DIR] + EXTRA)
print(f'\nthis session ran for {(time.time() - started) / 3600:.2f} h')

## 9. How far along is it?

The planner is the authority on what is left — it counts the files on Drive the
same way the sweep does. The grid below shows completed cells per dataset and
horizon (110 = 11 models × 10 seeds, with HAR-RV counting once per horizon).

In [ ]:
!python orchestrate/run_benchmark.py --results_dir $RESULTS_DIR \
    --anchor_dir $ANCHOR_DIR --scale raw_RV --dry_run 2>&1 | head -12

In [ ]:
import glob, os
import pandas as pd

paths = glob.glob(os.path.join(RESULTS_DIR, 'runs', '*', 'h*', '*.npz'))
if not paths:
    print('nothing on Drive yet')
else:
    done = pd.DataFrame([{'dataset': p.split(os.sep)[-3],
                          'horizon': p.split(os.sep)[-2]} for p in paths])
    print(f'{len(done)} cell(s) on Drive')
    display(done.value_counts().unstack(fill_value=0))

fail = os.path.join(RESULTS_DIR, 'failures.csv')
if os.path.exists(fail):
    f = pd.read_csv(fail)
    print(f'\n{len(f)} failed cell(s); by model:')
    display(f['model'].value_counts().to_frame('failures'))

## 10. The tables

`aggregate_results.py` re-scores whatever is on Drive — no retraining — so this
also works on a sweep that is still running. It says which blocks are still
short of models.

A raw sweep produces **no `MSE_ln` / `MAE_ln`**: those losses are the squared
and absolute error in `ln(RV)`, and the log of a forecast that can be negative
does not exist. The columns are kept in `metrics.csv` but stay empty, and no
`pivot_MSE_ln_*` file is written. `QLIKE`, `MSE_RV` and `MAE_RV` are the three
metrics this sweep reports, and they are exactly the three that are directly
comparable with the log sweep's (§12).

In [ ]:
!python orchestrate/aggregate_results.py --results_dir $RESULTS_DIR

In [ ]:
import os
import pandas as pd

TABLES = os.path.join(RESULTS_DIR, 'tables')

# mean +/- std over the seeds, per dataset and horizon
display(pd.read_csv(os.path.join(TABLES, 'metrics_mean.csv')).head(20))

# models x datasets, one metric, one horizon -- the shape a paper table has
for metric in ('QLIKE', 'MSE_RV'):
    for h in (1, 5, 22):
        path = os.path.join(TABLES, f'pivot_{metric}_h{h:02d}.csv')
        if os.path.exists(path):
            print(f'\n{metric}, h = {h}')
            display(pd.read_csv(path, index_col=0))

## 11. The raw-scale health check — non-positive forecasts

This section has no counterpart in the log notebook, because under `--log` it is
identically zero by construction: a log-scale forecast back-transforms to
`exp(.) > 0`. On the raw scale an unconstrained head — and HAR-RV's levels OLS
— can issue a **negative variance**, and the two losses handle it differently:

* `MSE_RV` / `MAE_RV` score against `max(forecast, 0)`, since a negative
  variance is not admissible and zero is the best feasible prediction the model
  could have issued;
* `QLIKE` is infinite at zero, so those forecasts are floored at
  `1e-4 × mean training RV` instead.

Both rules come from `HAR-RV_RUN.PY`'s raw branch, so the baseline and the deep
models are treated identically. But a model with a large share of floored
forecasts is being scored partly on the floor rather than on what it predicted,
and its QLIKE is a statement about the floor as much as about the model.

This is also the most likely place for the reused hyper-parameters to show. A
step size chosen on `ln(RV)` that is too large for a heavy-tailed target tends
to produce a model that overshoots into negative variance rather than one that
fails outright, and that model will look poor for a reason that has nothing to
do with its architecture. Read this table before the QLIKE ranking.

In [ ]:
import os
import pandas as pd

m = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'metrics.csv'))
total, floored = int(m['n_obs'].sum()), int(m['n_floored'].sum())
print(f'{floored} of {total} stored forecasts are <= 0 '
      f'({100 * floored / max(total, 1):.3f}%)')

by_model = m.groupby('model')[['n_floored', 'n_obs']].sum()
by_model['pct'] = (100 * by_model['n_floored'] / by_model['n_obs']).round(3)
display(by_model.sort_values('n_floored', ascending=False))

# target_dev is the other thing worth a glance: how far each cell's own actuals
# sit from Y^(h) rebuilt from the CSV, relative to the level of RV on the raw
# scale. ~1e-7 is the deep models' float32 round trip through the loader's
# scaler; anything larger means a cell was scored on the wrong rows.
print(f"\nlargest target deviation: {m['target_dev'].max():.2e}")

## 12. Raw against log, side by side

The comparison this notebook exists for. Because the hyper-parameters are
identical and only `--log` was removed, a difference between the two tables is
a difference of modelling scale and of nothing else.

Only the variance-scale metrics are shared, and that is not a limitation of the
tables: `exp()` of the log target is the raw target exactly, at every horizon,
so `QLIKE`, `MSE_RV` and `MAE_RV` are the same function of the same actuals in
both. `MSE_ln` has no raw counterpart at all.

Point `LOG_RESULTS` at the `ln(RV)` sweep's results directory. Negative
`delta_*` means the raw sweep loses less.

In [ ]:
import os
import pandas as pd

LOG_RESULTS = '/content/drive/MyDrive/ProjectC_benchmark'   # the ln(RV) sweep

log_path = os.path.join(LOG_RESULTS, 'tables', 'metrics_mean.csv')
if not os.path.exists(log_path):
    print('no log-scale tables at', log_path, '— skip this cell')
else:
    keys = ['dataset', 'horizon', 'model']
    shared = ['QLIKE', 'MSE_RV', 'MAE_RV']
    raw = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'metrics_mean.csv'))
    ln = pd.read_csv(log_path)
    both = raw[keys + shared].merge(ln[keys + shared], on=keys,
                                    suffixes=('_raw', '_ln'))
    for metric in shared:
        both[f'delta_{metric}'] = both[f'{metric}_raw'] - both[f'{metric}_ln']

    print(f'{len(both)} (dataset, horizon, model) cell(s) scored on both scales')
    display(both.head(20))

    # how often the raw scale wins, per horizon and per model
    wins = (both.assign(raw_wins=both['delta_QLIKE'] < 0)
                .groupby('horizon')['raw_wins'].agg(['sum', 'count']))
    wins.columns = ['raw lower QLIKE', 'cells']
    display(wins)
    display(both.groupby('model')['delta_QLIKE'].mean()
                .sort_values().to_frame('mean QLIKE, raw minus log'))

## 13. The DM / MCS inputs

`losses/<dataset>_h<hh>__<loss>__seed<S>.csv` is date-indexed, one column per
model, one row per forecast. A raw sweep writes three of them — `qlike`,
`se_rv`, `ae_rv` — where the log sweep writes five; `se_ln` and `ae_ln` are
simply absent, for the reason in §10. The mean of a column **is** the matching
cell in `metrics.csv`.

**Run the tests on the `__seedmean` files**: one series per model — the forecast
its ten repeats average to — so you get one DM statistic and one MCS, rather
than ten that cannot be pooled. In raw mode the modelling-scale and
variance-scale averages coincide, so the ensemble is the plain mean of the ten
forecasts.

At h > 1 the target windows of consecutive rows overlap by h−1 days, so the loss
differential is autocorrelated *by construction*: the DM long-run variance needs
a HAC estimator with at least h−1 lags, and the MCS block bootstrap needs a
block length that respects the same overlap.

In [ ]:
import glob, os
import pandas as pd

LOSSES = os.path.join(RESULTS_DIR, 'losses')
pick = sorted(glob.glob(os.path.join(LOSSES, '*__qlike__seedmean.csv')))
if not pick:
    print('no seedmean matrices yet — they appear once a block has >1 seed')
else:
    path = pick[0]
    print(os.path.basename(path))
    L = pd.read_csv(path, index_col=0, parse_dates=True)
    print(L.shape, '(forecasts x models)')
    display(L.head())

    # HAR-RV minus each model, row by row: positive = the model loses less.
    # rsub with axis=0 broadcasts down the dates; a plain Series - DataFrame
    # would align on the column labels and give an all-NaN frame.
    if 'HAR-RV' not in L.columns:
        print('no HAR-RV column in this block — nothing to compare against')
    else:
        d = L.drop(columns='HAR-RV').rsub(L['HAR-RV'], axis=0)
        display(d.mean().sort_values(ascending=False)
                 .to_frame('mean QLIKE saved vs HAR-RV'))

## 14. Download the results (optional)

They are already on Drive. This is for pulling the small files — the raw
anchors, the tables and the loss matrices, not the 3 000 `.npz` — onto your
laptop in one archive.

In [ ]:
import glob, os, zipfile

out = '/content/ProjectC_benchmark_raw_tables.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ('tables', 'forecasts', 'losses'):
        for root, _, names in os.walk(os.path.join(RESULTS_DIR, sub)):
            for name in names:
                path = os.path.join(root, name)
                z.write(path, os.path.relpath(path, RESULTS_DIR))
    for name in ('manifest.json', 'failures.csv'):
        path = os.path.join(RESULTS_DIR, name)
        if os.path.exists(path):
            z.write(path, name)
    # the anchors this sweep was run from -- without them the table is not
    # reproducible, and they are a few kB
    for path in sorted(glob.glob(os.path.join(ANCHOR_DIR, '*_best.json'))):
        z.write(path, os.path.join('anchors_raw', os.path.basename(path)))

print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')

from google.colab import files
files.download(out)

## Notes

* **What differs from the log benchmark.** One token, `--log`, removed from ten
  command lines. No file in the repository is edited, no hyper-parameter is
  changed, no flag of `run_benchmark.py` changes meaning; the target, the
  losses, HAR-RV's own invocation and the QLIKE floor all follow from that one
  absence. `--scale raw_RV` on every call is the check that they did.
* **Nothing is tuned here.** The hyper-parameters are the ln(RV) Optuna
  winners for **EUR/USD at h = 1**, applied to every dataset, every horizon and
  now to a second modelling scale. Two transfers stacked on one another — worth
  one sentence wherever these tables are reported. Re-tuning under `--raw` is
  `tuning/optuna_tune.py --model <M> --raw --out_dir <dir>`, one flag, if the
  question ever becomes "what is the best each architecture can do on raw RV"
  rather than "what does the scale change".
* **Resuming.** Re-run steps 2–5 (step 5 is a second and is idempotent), then
  step 8. A cell whose `.npz` is on Drive is skipped, and a failed cell is
  recorded in `failures.csv` and skipped too — `['--retry_failed']` in `EXTRA`
  re-runs those.
* **Non-positive forecasts are the raw scale's characteristic failure mode**
  and are not an error: §11 counts them, `metrics.csv` carries `n_floored` per
  cell, and the aggregation prints a warning when any exist. Under `--log`
  there are none, so the two sweeps are not on equal footing here — a raw model
  is spending some of its QLIKE on the floor.
* **Which metrics exist.** `QLIKE`, `MSE_RV`, `MAE_RV`. `MSE_ln` and `MAE_ln`
  are empty columns and no `pivot_MSE_ln_*` is written. The three that exist are
  comparable with the log sweep's, because `exp()` of the log target is the raw
  target exactly at every horizon.
* **The training budget is not the tuning budget.** The studies searched at
  `--train_epochs 30 --patience 7`; the sweep runs every cell at 50 / 10,
  because `train_epochs` and `patience` are flags the orchestrator owns. That is
  the same mismatch the log benchmark has, kept deliberately so the two are
  comparable — but note that a `--lradj cosine` anchor divides by
  `train_epochs` to shape its schedule, so for those cells the cap is not merely
  a stopping point.
* **Drive is slow for many small files.** The sweep writes one `.npz` per cell
  (3 000 of them) and the aggregation a few thousand CSVs. That is fine, but a
  §10 run takes a minute or two once the grid is full.
* **Cost.** FITS, DLinear and TSLANet are seconds per cell; TimesNet and MSGNet
  dominate. Plan on several sessions for the full 3 000, or start with
  `['--itr', '3']`.
* Details, file formats and the two `exp/` changes this work made:
  `orchestrate/README.md`.